# Traditional Solution

### 1. Load and Clean the Dataset

In [ ]:
import pandas as pd

In [ ]:
# Load the dataset
df = pd.read_csv("US_Accidents_March23.csv")

In [ ]:
import os
import math
import random
import csv
from typing import Dict, Any, Optional, List, Iterable, Tuple

US_STATES = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA",
    "ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK",
    "OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
}

WEATHER_BOUNDS = {
    "Temperature(F)": (-70.0, 130.0),
    "Wind_Chill(F)": (-50.0, 130.0),
    "Humidity(%)": (0.0, 100.0),
    "Pressure(in)": (15.0, 32.0),
    "Visibility(mi)": (0.0, 100.0),
    "Wind_Speed(mph)": (0.0, 200.0),
    "Precipitation(in)": (0.0, 50.0),
}

COLS_TO_DROP = {"ID", "Source", "Zipcode", "Timezone", "Airport_Code", "End_Lat", "End_Lng"}

WIND_DIR_MAP = {
    "VARIABLE": "VAR",
    "VAR": "VAR",
    "CALM": "CALM",
    "NORTH": "N",
    "SOUTH": "S",
    "EAST": "E",
    "WEST": "W"
}

TWILIGHT_COLS = (
    "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"
)

BOOL_COLS = (
    "Amenity","Bump","Crossing","Give_Way","Junction","No_Exit","Railway",
    "Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal","Turning_Loop"
)

def clean_us_accidents(
    rows: Iterable[Dict[str, Any]],
    header_cols: List[str],
    output_csv_path: str,
) -> None:
    weather_cols = [c for c in WEATHER_BOUNDS.keys() if c in header_cols]

    def clean_rows(row: Dict[str, Any]) -> Optional[Tuple[str, Dict[str, Any]]]:
        row = dict(row)

        # Trim strings
        for k, v in row.items():
            if isinstance(v, str):
                row[k] = v.strip()

        # Clean start time
        st = row.get("Start_Time")
        if not st or (isinstance(st, str) and not st.strip()):
            return None

        # Bound severity from 1 to 4
        if "Severity" in row:
            sev_val = int(round(float(row["Severity"])))
            if not (1 <= sev_val <= 4):
                return None
            row["Severity"] = sev_val

        # Bound coordinates
        lat = float(row.get("Start_Lat"))
        if not (-90.0 <= lat <= 90.0):
            return None
        row["Start_Lat"] = lat
        lng = float(row.get("Start_Lng"))
        if not (-180.0 <= lng <= 180.0):
            return None
        row["Start_Lng"] = lng

        # Remove all states not in US_STATES
        if "State" in row:
            state = row["State"]
            if state and isinstance(state, str):
                state = state.upper()
                row["State"] = state if state in US_STATES else None
            else:
                row["State"] = None

        # Bound weather
        for c, (lo, hi) in WEATHER_BOUNDS.items():
            if c in row:
                val = row[c]
                if val is None or val == "":
                    row[c] = None
                else:
                    try:
                        fval = float(val)
                        row[c] = fval if (lo <= fval <= hi) else None
                    except (TypeError, ValueError):
                        row[c] = None

        # Standardize wind direction
        if "Wind_Direction" in row:
            wd = row["Wind_Direction"]
            if wd and isinstance(wd, str):
                row["Wind_Direction"] = WIND_DIR_MAP.get(wd.upper(), wd.upper())
            else:
                row["Wind_Direction"] = None

        # Standardize twilight columns
        for c in TWILIGHT_COLS:
            if c in row:
                val = row[c]
                if val and isinstance(val, str):
                    val_t = val.title()
                    row[c] = val_t if val_t in ("Day", "Night") else "Unknown"
                else:
                    row[c] = "Unknown"

        # Assume missing boolean to be false
        for c in BOOL_COLS:
            if c in row:
                v = row[c]
                if isinstance(v, bool):
                    pass
                elif isinstance(v, str):
                    row[c] = True if v.lower() == "true" else False
                else:
                    row[c] = False

        # Extract ID before dropping it
        rid = str(row.get("ID", "") or "").strip()

        # Drop redundant columns
        for c in COLS_TO_DROP:
            row.pop(c, None)

        return (rid, row)

    # Map
    mapped: List[Tuple[str, Dict[str, Any]]] = []
    for row in rows:
        out = clean_rows(row)
        if out is not None:
            mapped.append(out)

    # Reduce
    deduped_dict: Dict[str, Dict[str, Any]] = {}
    for rid, row in mapped:
        if rid not in deduped_dict:
            deduped_dict[rid] = row

    cleaned: List[Dict[str, Any]] = list(deduped_dict.values())

    # Sample weather
    medians: Dict[str, Optional[float]] = {}
    if weather_cols and cleaned:
        sample_size = min(50000, len(cleaned))
        rng = random.Random(1)
        sampled_rows = rng.sample(cleaned, sample_size)

        sampled_weather = [tuple(r.get(c) for c in weather_cols) for r in sampled_rows]

        for i, c in enumerate(weather_cols):
            vals = []
            for w in sampled_weather:
                v = w[i]
                if v is not None:
                    fv = float(v)
                    if not math.isnan(fv):
                        vals.append(fv)

            if not vals:
                medians[c] = None
            else:
                vals.sort()
                n = len(vals)
                mid = n // 2
                medians[c] = vals[mid] if n % 2 == 1 else (vals[mid - 1] + vals[mid]) / 2.0

    # Fill missing weather with median
    for r in cleaned:
        for c in weather_cols:
            if r.get(c) is None and medians.get(c) is not None:
                r[c] = float(medians[c])

    # Write to CSV
    os.makedirs(os.path.dirname(output_csv_path) or ".", exist_ok=True)

    output_cols = [c for c in header_cols if c not in COLS_TO_DROP]

    # Include any extra columns that may exist in cleaned rows but not header_cols
    extra_cols = []
    seen = set(output_cols)
    for r in cleaned:
        for c in r.keys():
            if c not in seen:
                extra_cols.append(c)
                seen.add(c)

    output_cols.extend(extra_cols)

    with open(output_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=output_cols, extrasaction="ignore")
        writer.writeheader()
        for r in cleaned:
            writer.writerow(r)

In [ ]:
# Traditional Cleaning of Dataset
rows = df.to_dict(orient="records")

clean_us_accidents(
    rows=rows,
    header_cols=list(df.columns),
    output_csv_path="cleaned_accidents.csv"
)

In [ ]:
# Load cleaned dataframe
df_clean = pd.read_csv('cleaned_accidents.csv')

### 2. Analysis of Time of the accident

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import calendar

In [ ]:
# Load in the dataset into a panda dataframe (note it is big so will take some time)
df = df_clean.copy()

In [ ]:
# Change start time to datetime type for
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")

#### This bar chart shows the amount of car crashes per hour of day.
From this we see that most crashes occur between 6-8 am and 3 - 6 pm. These are traditionally considered rush hour, when most people are commuting to and from work and school. This is usually when roads are busiest and reflects in this data as that is when there are the most amount of car crashes.

In [ ]:
# Histogram for crashes on every hour of day
fig, ax = plt.subplots(figsize=(15,10))

sns.histplot(
    df['Start_Time'].dt.hour,
    bins = 24,
    discrete = True,
    ax = ax
)

ax.set_xticks(range(24))
ax.ticklabel_format(style = 'plain', axis = 'y')

plt.grid(axis = 'y')

plt.xlabel("Hour of Day")
plt.ylabel("Number of Occurrence")
plt.title('Accidents Count By Time of Day')

plt.show()

#### Distribution of crashes based on the year
This shows that the number of crashes is increasing each year. (Note that 2023 is an outlier as the data collection stops in march, 2016 is a slihght outlier as it only starts to collect data from February)

In [ ]:
# Horizontal bar chart showing distrbution of crashes through the years
year_counts = df['Start_Time'].dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize =( 15,10))

year_counts.plot(kind = 'barh', ax = ax)

ax.set_ylabel("Year")
ax.set_xlabel("Number of Occurrence")
ax.set_title('Accidents Distribution by Year')

ax.ticklabel_format(style = 'plain', axis = 'x')

plt.grid(axis = 'x')

plt.show()

#### This graph shows the distribution of car crashes based on the day of the week.
From this we can see that the weekend has less crashes, I would presume this is because there are less cars on the road as less people are communiting to and from work/school.

In [ ]:
# Count accidents by day of week horizontal bar chart
day_counts = df['Start_Time'].dt.day_name().value_counts()

# Reorder to normal week order
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = day_counts.reindex(day_order)

fig, ax = plt.subplots(figsize = (15,10))

day_counts.plot(kind = 'barh', ax = ax)

ax.set_ylabel("Day of Week")
ax.set_xlabel("Number of Occurrence")
ax.set_title('Accidents Distribution by Day of Week')

ax.ticklabel_format(style = 'plain', axis = 'x')

plt.grid(axis = 'x')

plt.show()

### 3. Analysis of Points of Interest at location of Accident

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Read in the dataset (500,000 smaller version)
df = df_clean[["Amenity","Bump","Crossing","Give_Way","No_Exit","Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal","Turning_Loop","Severity"]].copy()
dfs = df_clean[["State","Start_Time","End_Time","Severity"]].copy()

####
Amenity     -       Refers to particular places such as restaurant, library, college, bar, etc.
Bump        -       Refers to speed bump or hump to reduce the speed.
Crossing    -       Refers to any crossing across roads for pedestrians, cyclists, etc.
Give-way    -       A sign on road which shows priority of passing.
Junction    -       Refers to any highway ramp, exit, or entrance.
No-exit     -       Indicates there is no possibility to travel further by any transport mode along a formal path or route.
Railway     -       Indicates the presence of railways.
Roundabout  -       Refers to a circular road junction.
Station     -       Refers to public transportation station (bus, metro, etc.).
Stop        -       Refers to stop sign.
Traffic Calming -   Refers to any means for slowing down traffic speed.
Traffic Signal -    Refers to traffic signal on intersections.
Turning Loop    -   Indicates a widened area of a highway with a non-traversable island for turning around.

In [ ]:
# Show accidents counts by proximity to each POI
poi_cols = ["Amenity","Bump","Crossing","Give_Way","No_Exit","Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal","Turning_Loop"]
counts = df[poi_cols].sum().sort_values(ascending=False)
# Create the plot
plot_axis = counts.plot(
    kind="bar",
    figsize=(20, 5),
    fontsize=13,
    color="coral",
    width=0.9,
    rot=0
)

# Customise the axes and title
plot_axis.set_ylim((0, 1200000))
plot_axis.set_xlabel("POI", fontsize=13)
plot_axis.set_ylabel("Number of Accidents", fontsize=13)
plot_axis.set_title("Accident by proximity to POI", fontsize=13)
plt.show()

##### We see then that the majority of accidents take place at junctions. This is indicated by Traffic_signal, Crossing, and Stop. With all other POIs contributing a small number. We will now look at the top 5 POI features by severity to see for any patterns.

In [ ]:
colors = ['#fee5d9', '#fcae91', '#fb6a4a', '#de2d26', '#a50f15']

severity_cols = ["Traffic_Signal","Crossing","Stop","Station","Amenity"]

plot_data = df.groupby("Severity", observed=True)[poi_cols].sum().T

ax = plot_data.plot(kind="bar", figsize=(20,6), color=colors)
ax.set_xlabel("POI")
ax.set_ylabel("Count of True values")
ax.set_title("Top 5 POIs by Severity")
plt.xticks(rotation=0)
plt.legend('',frameon=False)
plt.show()

### 4. Analysis of Weather Conditions of Accidents

In [ ]:
df = df_clean.copy()
def plot_histogram(col):
    num_bins = 30
    min_col = df[col].min()
    max_col = df[col].max()
    width = (max_col - min_col) / num_bins

    bin_counts = {}
    for index, row in df.iterrows():
        value = row[col]
        
        # Manual Binning Logic
        if not pd.isna(value):
            bin_idx = int((value - min_col) / width)
            if bin_idx >= num_bins:
                bin_idx = num_bins - 1
            
            bin_counts[bin_idx] = bin_counts.get(bin_idx, 0) + 1

    final_counts = [bin_counts.get(i, 0) for i in range(num_bins)]
    final_edges = [min_col + (i * width) for i in range(num_bins)]

    plt.figure(figsize=(8, 5))
    plt.bar(final_edges, final_counts, width=width)
    plt.xlabel(col)
    plt.ylabel('Number of Accidents')
    plt.title(f'Distribution of Accidents by {col}')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_histogram('Temperature(F)')

In [ ]:
plot_histogram('Wind_Chill(F)')

In [ ]:
plot_histogram("Humidity(%)")

In [ ]:
plot_histogram("Pressure(in)")

In [ ]:
plot_histogram("Visibility(mi)")

In [ ]:
plot_histogram("Visibility(mi)")
wind_counts_dict = {}
for index, row in df.iterrows():
    direction = row['Wind_Direction']

    if pd.notna(direction):
        direction = str(direction).strip()
        wind_counts_dict[direction] = wind_counts_dict.get(direction, 0) + 1

wind_direction = pd.Series(wind_counts_dict).sort_values(ascending=True)
plt.figure(figsize=(10, 8))
wind_direction.plot(kind='barh')

plt.title('Wind Direction Distribution', fontsize=14)
plt.xlabel('Count', fontsize=12)
plt.ylabel('Wind Direction', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plot_histogram("Wind_Speed(mph)")

In [ ]:
plot_histogram("Precipitation(in)")

In [ ]:
detailed_categories = {
    'Rainy': ['rain', 'rainy', 'drizzle', 'shower', 'raining', 'rainfall', 'precipitation'],
    'Cloudy': ['cloud', 'cloudy', 'overcast', 'mostly cloudy', 'partly cloudy'],
    'Clear': ['clear', 'sunny', 'fair'],
    'Snowy': ['snow', 'snowy', 'blizzard', 'sleet', 'ice', 'wintry', 'hail'],
    'Foggy': ['fog', 'foggy', 'mist', 'haze', 'smoke', 'dust', 'ash', 'sand'],
    'Windy': ['wind', 'windy', 'breezy'],
    'Storm': ['storm', 'thunder', 'lightning', 'squall', 'tornado']
}

def detailed_categorize(condition):
    if pd.isna(condition):
        return 'Unknown'
    condition_lower = str(condition).lower()
    for category, keywords in detailed_categories.items():
        if any(keyword in condition_lower for keyword in keywords):
            return category
    return 'Other'

weather_counts_dict = {}
for index, row in df.iterrows():
    condition = row['Weather_Condition']
    category = detailed_categorize(condition)
    weather_counts_dict[category] = weather_counts_dict.get(category, 0) + 1

weather_category = pd.Series(weather_counts_dict).sort_values(ascending=True)
plt.figure(figsize=(10, 8))
weather_category.plot(kind='barh')
plt.title('Weather Condition Distribution', fontsize=14)
plt.xlabel('Count', fontsize=12)
plt.ylabel('Weather Condition', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Box-plot with severity
import seaborn as sns
weather_cols = [
    'Temperature(F)', 'Wind_Chill(F)',
    'Humidity(%)', 'Pressure(in)', 
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, var in zip(axes.flatten(), weather_cols):
    sns.boxplot(x='Severity', y=var, data=df, ax=ax)
    ax.set_title(f'{var} by Severity Level')
    ax.set_xlabel('Severity')
    ax.set_ylabel(var)

plt.tight_layout()
plt.show()

### 5. Create a Map Distribution of the locations for each Accident

In [ ]:
df = df_clean[["Start_Lat", "Start_Lng","City","State"]].copy()

#### We will be using Start_Lat and Start_Lng

This is because we the column End_Lat and End_lng is filled with Null.


In [ ]:
# Validating coordinates for US bounds

print("Latitude range:", df["Start_Lat"].min(),"to", df["Start_Lat"].max())
print("Longitude range:", df["Start_Lng"].min(),"to", df["Start_Lng"].max())

#### We will be using a sample Data Frame with just 50000 rows.
 We use plotly.express to show our geographical analysis.

In [ ]:
!pip install plotly

In [ ]:
import plotly.express as px

state_counts = df.groupby("State").size().reset_index(name="Accident_Count")
sample_df = df.sample(50000, random_state=42)
state_counts.head()

#### Analysis
Bellow we can see that the state of Califoria has highes accident count (1.7 million) accourding to our Sample Data, then Florida with 0.9 million accidemnts and third is Texas with 0.6 million accidents.

In [ ]:
fig = px.choropleth(
    state_counts,
    locations="State",
    locationmode="USA-states",
    color="Accident_Count",
    scope="usa",
    color_continuous_scale=["yellow", "red","purple"],
    title="Number of Accidents per State"
)

fig.update_layout(height=600)

fig.show()

#### Scatter Map
Our scatter map shows that most accidents occur on the East side and west side of the country while the middle is not as dense.

In [ ]:
fig = px.scatter_mapbox(
    sample_df,
    lat="Start_Lat",
    lon="Start_Lng",
    zoom=3,
    mapbox_style="carto-positron",
    color_discrete_sequence=["red"],
    hover_data=["Start_Lat", "Start_Lng"],
    title="Accident Locations (Sample)"
)

fig.update_layout(height=600)
fig.show()

#### Hotspots
Bellow is top 10 and least 10 accident counts accross all Lat/Lng pair. the table shows real hotspots the top one is 25.9, -80.2 Miami, LA and NYC.

In [ ]:
# Rounding the values to 1 decimal place

df["Lat_Rounded"] = df["Start_Lat"].round(1)
df["Lng_Rounded"] = df["Start_Lng"].round(1)

# We count accidents per Lat/Lng

lat_lng_counts = (
    df.groupby(["Lat_Rounded", "Lng_Rounded"])
      .size()
      .reset_index(name="Accident_Count")
)

#### Spatial Analysis Findings From Tables Below

The highest accident density occurs around latitude 25.9 and longitude -80.2, corresponding to the Miami metropolitan area. Other high-density regions include Los Angeles and New York City.

This suggests that accident frequency correlates strongly with high urban density and major metropolitan traffic hubs.

In [ ]:
# Top 10 highest density locations

top_locations = lat_lng_counts.sort_values(
    "Accident_Count",
    ascending=False
).head(10)

top_locations

In [ ]:
# Lowest locations but not 0

lowest_locations = lat_lng_counts[
    lat_lng_counts["Accident_Count"] > 0
].sort_values("Accident_Count").head(10)

lowest_locations

#### Latitude distribution
The following distribiution tells us which latitude zones are most dangerous.

34-35 -> Southern California

40-41 -> NYC / Northeast

29-30 -> Texas / Gulf

In [ ]:
df["Lat_Rounded"] = df["Start_Lat"].round(1)

lat_counts = df.groupby("Lat_Rounded").size().reset_index(name="Count")

import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(lat_counts["Lat_Rounded"], lat_counts["Count"])
plt.title("Accident Distribution by Latitude (Rounded)")
plt.xlabel("Latitude")
plt.ylabel("Number of Accidents")
plt.show()

In [ ]:
# Finding highest latitude bands
lat_counts.sort_values("Count", ascending=False).head(10)

#### Longitude distribution
The following distribiution tells us which longitude zones are most dangerous.

Highest density around -80.2 -> Miami area

Strong cluster around -118.2 / -118.3

This shows that accident density is heavily concentrated in major coastal metropolitan regions.

In [ ]:
df["Lng_Rounded"] = df["Start_Lng"].round(1)

lng_counts = df.groupby("Lng_Rounded").size().reset_index(name="Count")

import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(lng_counts["Lng_Rounded"], lng_counts["Count"])
plt.title("Accident Distribution by Longitude (Rounded)")
plt.xlabel("Longitude")
plt.ylabel("Number of Accidents")
plt.show()

In [ ]:
# Finding highest longitude bands
lng_counts.sort_values("Count", ascending=False).head(10)